### CRISP-DM Phase 2 - Data Understanding : Earth Observation

Exploratory analysis of selected datasets from **Climate Data Store** by *Copernicus Climate Change Service*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from iso3166 import countries_by_alpha3
from matplotlib.colors import ListedColormap

In [ ]:
# Load the datasets
temperature = pd.read_csv('data/2m_temperature.csv')
wind = pd.read_csv('data/instantaneous_wind_gust.csv')
sea = pd.read_csv('data/sea_level_anomaly.csv')
snowmelt = pd.read_csv('data/snowmelt.csv')
spei = pd.read_csv('data/spei.csv')
precipitation = pd.read_csv('data/total_precipitation.csv')

var_dict = {'2m temperature': {'data': temperature, 'unit': 'K'}, 
            'Instantaneous wind gust': {'data': wind, 'unit': 'm/s'}, 
            'Sea level anomaly': {'data': sea, 'unit': 'm'}, 
            'Snowmelt': {'data': snowmelt, 'unit': 'mm of water equivalent'}, 
            'SPEI': {'data': spei, 'unit': 'dimensionless'}, 
            'Total precipitation': {'data': precipitation, 'unit': 'mm'}}

sensor = pd.read_csv('data/sensor_bis.csv')
law_classified = pd.read_csv('../law/data/law_classified.csv')

DU1 - Attributes

In [ ]:
for name, details in var_dict.items():
    df = details['data']
    print(f"Dataset: {name}\nShape: {df.shape}\nColumns: {df.columns.tolist()}")
    print(f"Data types:\n{df.dtypes}")

DU2 - Statistical properties

In [ ]:
## Geographic Coverage
countries = {}    
for name, details in var_dict.items():
    df = details['data']
    countries[name] = set(df['country'].dropna().unique())
    print(f"Total countries in {name}: {len(countries[name])}")
print(f"Countries in all datasets: {len(set.intersection(*countries.values()))}")

In [ ]:
# Overlap with law dataset
law_countries = set(law_classified['Country'].dropna().unique())
sensor_countries = set(sensor['Country'].dropna().unique())
print(f"Countries in law and satellite datasets: {len(law_countries & sensor_countries)}")
print(f"Countries in law only: {len(law_countries - sensor_countries)}\n{law_countries - sensor_countries}")
print(f"Countries in satellite only: {len(sensor_countries - law_countries)}\n{sensor_countries - law_countries}")

In [ ]:
## Temporal Coverage
years = {}
for name, details in var_dict.items():
    df = details['data']
    years[name] = pd.to_datetime(df['time'], errors='coerce').dt.year.astype('Int64')
    print(f"{name} year range : {years[name].min()} - {years[name].max()}")

In [ ]:
## Explore variables
for name, details in var_dict.items():
    df = details['data']
    print(f"{name} statistics:\n{df['value'].describe().round(5)}")

DU3 - Data quality

In [ ]:
## Missing Values
print(sensor.isna().sum())
print((sensor.isna().sum() / len(sensor) * 100).round(2))

DU4 - Visual Exploration

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, (name, details) in enumerate(var_dict.items()):
    df = details['data']
    df['value'].dropna().hist(ax=axes[i], bins=50, edgecolor='white')
    
    axes[i].set_title(name)
    axes[i].set_xlabel(f"Value ({details['unit']})")
plt.tight_layout()
plt.savefig('outputs/3_value_distribution.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, (name, details) in enumerate(var_dict.items()):
    df = details['data']
    df['time'] = pd.to_datetime(df['time'])
    df['year'] = df['time'].dt.year
    
    annual = df.dropna(subset=['year', 'value']).groupby('year')['value'].mean()
    axes[i].plot(annual.index, annual.values)
    axes[i].set_title(name)
    axes[i].set_xlabel('Year')
    axes[i].set_ylabel(f"Mean value ({details['unit']})")
plt.tight_layout()
plt.savefig('outputs/3_value_trend.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
sensor_countries = set(sensor['Country'].str.strip().unique())
law_countries = set(law_classified['Country'].str.strip().unique())

def coverage_category(iso):
    in_sensor = iso in sensor_countries
    in_law = iso in law_countries
    if in_sensor and in_law:
        return 'Both'
    elif in_sensor:
        return 'Sensor only'
    elif in_law:
        return 'Law only'

world = gpd.read_file("https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip")
iso_fixes = {'France': 'FRA', 'Norway': 'NOR', 'South Sudan': 'SSD', 'Kosovo': 'XKX', 
             'Northern Cyprus': 'CYN', 'Somaliland': 'SOM'}
for country_name, correct_iso in iso_fixes.items():
    world.loc[world['NAME'] == country_name, 'ISO_A3'] = correct_iso
categories_order = ['Both', 'Sensor only', 'Law only']
world['coverage'] = world['ISO_A3'].apply(coverage_category)
world['coverage'] = pd.Categorical(world['coverage'], categories=categories_order, ordered=True)

custom_palette = ListedColormap(['lightgrey', 'blue', 'green'])
fig, ax = plt.subplots(1, 1, figsize=(15, 8))
world.plot(column='coverage', categorical=True, cmap=custom_palette, edgecolor='white', linewidth=0.3, 
           ax=ax, legend=True, missing_kwds={'color': 'red', 'label': 'Neither'},
           legend_kwds={'loc': 'center left','frameon': True})
ax.axis('off')
plt.tight_layout()
plt.savefig('outputs/3_overlap_map.png', dpi=150, bbox_inches='tight')
plt.close()

DU5 - Ethical Concerns, Risks, and Biases

In [ ]:
sensor_countries = sensor[['Country', 'Country_name']].drop_duplicates()
sensor_merged = world.merge(sensor_countries.rename(columns={'Country': 'ISO_A3'}), on='ISO_A3', how='left')
sensor_merged['area_km2'] = sensor_merged['geometry'].to_crs('ESRI:54009').area / 1e6

# Count observations per country 
count = sensor.groupby('Country').size().reset_index(name='count')
sensor_merged = sensor_merged.merge(count.rename(columns={'Country': 'ISO_A3'}), on='ISO_A3', how='left') 

# Population and country size comparison
pop_correlation = sensor_merged[['count', 'POP_EST']].dropna().corr().iloc[0,1]
size_correlation = sensor_merged[['count', 'area_km2']].dropna().corr().iloc[0,1]

print(f"Correlation between observation count and population: {pop_correlation.round(2)}")
print(f"Correlation between observation count and country size: {size_correlation.round(2)}")